# High-Danger Cycle Analysis: Trends, Contrasts, and Anomalies

Comprehensive analysis across 4 zones x 8 high-danger dates (32 scene-date combos).

Throughout this notebook we compare **`d_empirical`** (z-score from ECDF, can go negative)
against **`combined_distance`** (blends ML Mahalanobis components, stays mostly positive)
to see how much the ML-based metric reduces weather-driven bias.

---

## Top-Level Findings

### 1. SWE / snow depth biases *both* distance metrics, but empirical far more
- **d_empirical vs SWE**: r = -0.80 — deep snowpacks push the entire distribution negative
- **combined_distance vs SWE**: r = -0.57 — ML blending reduces the bias but doesn't eliminate it
- **Mahalanobis vs SWE**: r = -0.46 — most stable, but still correlated
- The empirical metric swings from -1.15 to +0.57 across dates; combined_distance from
  -0.46 (Soldier, Apr 2023) to +1.71 (Soldier, Feb 2024) — still a 3x range

### 2. Two distinct failure modes
- **Wet snow (Apr 2023)**: uniform scene-wide drop, 47% of pixels below -1.0 z-score,
  low orbit spread (0.32). combined_distance drops to 0.53 (vs ~1.3 typical) — ML helps but
  doesn't fully compensate
- **Storm loading (Mar 2023)**: orbit 93 dominates, high variance. combined_distance
  drops to 0.56 and Soldier goes *negative* (-0.02) — the ML model also struggles here

### 3. The Soldier zone amplifies everything
- Most extreme values in both metrics: d_emp ranges [-1.67, +0.95], combined_distance [-0.46, +1.71]
- Largest orbit spreads (1.36 on Mar 13). Lower elevation, more exposed terrain?

### 4. 2024-02-05 is the standout detection event
- 90,639 total detections — 3-10x more than any other date
- Both metrics agree: d_emp +0.57, combined_distance 1.41, Mahalanobis 0.19
- Moderate warming (Tmax 6.1C), low SWE (10.4") — the "sweet spot" for detection

### 5. combined_distance is more stable than empirical, but not immune
- Empirical CV across dates: very high | Combined CV: lower but still significant
- On good dates (winter, low SWE), combined_distance ~1.3-1.5 across all zones
- On bad dates (Mar/Apr 2023), it drops to 0.5 or below — still a 3x swing

### 6. Detection elevation shifts with season
- Dec dates: mean detection elevation ~2100-2175m
- Feb-Apr dates: ~2360-2560m — detections migrate uphill as snow line rises

### 7. VV consistently drops more than VH
- VV-VH difference averages -0.24 across all dates (VV more negative)
- Smallest gap on Apr 1 (-0.08) — wet snow attenuates both polarizations equally

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

OUT_DIR = Path('/Users/zmhoppinen/Documents/sarvalanche/scripts/issw_analysis/snfac_high_dangers')
df = pd.read_parquet(OUT_DIR / 'stats.parquet')
snotel = pd.read_parquet(OUT_DIR / 'snotel_combined.parquet')

# Short zone labels for plotting
zone_short = {
    'Banner_Summit': 'Banner',
    'Galena_Summit_&_Eastern_Mtns': 'Galena',
    'Sawtooth_&_Western_Smoky_Mtns': 'Sawtooth',
    'Soldier_&_Wood_River_Valley_Mtns': 'Soldier',
}
df['zone_short'] = df['zone'].map(zone_short)

dates = sorted(df['date'].unique())
print(f'{len(df)} rows: {df["zone"].nunique()} zones x {len(dates)} dates')
print('Dates:', dates)

## 1. Scene-Level Backscatter Distance: The Big Picture

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('d_emp_mean', 'Empirical Distance Mean'),
    ('d_mah_mean', 'Mahalanobis Distance Mean'),
    ('d_comb_mean', 'Combined Distance Mean'),
]

for ax, (col, title) in zip(axes, metrics):
    pivot = df.pivot_table(values=col, index='date', columns='zone_short')
    pivot.plot(kind='bar', ax=ax, edgecolor='black', lw=0.5, width=0.75)
    ax.axhline(0, color='black', ls='-', lw=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('z-score' if 'emp' in col or 'comb' in col else 'distance')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=7, title='Zone')
    ax.grid(True, alpha=0.2, axis='y')

fig.suptitle('Scene-level distances across all dates and zones', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. SWE Controls Distance Bias — Empirical Most, Combined Less, Mahalanobis Least

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

weather_vs_sar = [
    # Top row: SWE vs all three metrics
    ('swe_at_date', 'd_emp_mean', 'SWE (in)', 'd_empirical mean'),
    ('swe_at_date', 'd_comb_mean', 'SWE (in)', 'combined_distance mean'),
    ('swe_at_date', 'd_mah_mean', 'SWE (in)', 'Mahalanobis mean'),
    # Bottom row: warm days vs all three metrics
    ('days_tmax_above_0', 'd_emp_mean', 'Days Tmax > 0C (±14d)', 'd_empirical mean'),
    ('days_tmax_above_0', 'd_comb_mean', 'Days Tmax > 0C (±14d)', 'combined_distance mean'),
    ('days_tmax_above_0', 'd_mah_mean', 'Days Tmax > 0C (±14d)', 'Mahalanobis mean'),
]

for ax, (wx, sy, xlabel, ylabel) in zip(axes.flat, weather_vs_sar):
    valid = df[[wx, sy, 'zone_short', 'date']].dropna()
    for zone, grp in valid.groupby('zone_short'):
        ax.scatter(grp[wx], grp[sy], s=50, label=zone, alpha=0.8, edgecolors='black', lw=0.5)
    for _, r in valid.iterrows():
        ax.annotate(r['date'][5:], (r[wx], r[sy]), fontsize=5, alpha=0.6,
                    textcoords='offset points', xytext=(3, 3))
    r_val = valid[wx].corr(valid[sy])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} (r = {r_val:.2f})', fontsize=10)
    ax.axhline(0, color='black', ls='--', lw=0.5, alpha=0.5)
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    'Weather bias: empirical (r=-0.80) > combined (r=-0.57) > Mahalanobis (r=-0.46)\n'
    'ML blending reduces but does NOT eliminate the SWE/temperature confound',
    fontsize=12, y=1.03,
)
plt.tight_layout()
plt.show()

## 3. Two Failure Modes: Wet Snow vs Storm Loading

In [ ]:
by_date = df.groupby('date').agg({
    'd_emp_mean': 'mean',
    'd_emp_std': 'mean',
    'orbit_emp_VV_spread': 'mean',
    'd_emp_frac_lt_-1.0': 'mean',
    'd_emp_frac_gt_1.0': 'mean',
    'days_tmax_above_0': 'first',
    'swe_at_date': 'first',
}).rename(columns={
    'd_emp_frac_lt_-1.0': 'frac_lt_m1',
    'd_emp_frac_gt_1.0': 'frac_gt_1',
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Scatter: orbit spread vs d_emp_mean
ax = axes[0, 0]
colors = ['tab:red' if d in ['2023-04-01', '2023-03-13'] else 'tab:blue' for d in by_date.index]
ax.scatter(by_date['d_emp_mean'], by_date['orbit_emp_VV_spread'],
           c=colors, s=100, edgecolors='black', lw=0.5, zorder=5)
for d in by_date.index:
    ax.annotate(d, (by_date.loc[d, 'd_emp_mean'], by_date.loc[d, 'orbit_emp_VV_spread']),
                fontsize=8, textcoords='offset points', xytext=(5, 5))
ax.set_xlabel('d_empirical mean')
ax.set_ylabel('Orbit VV spread (max - min)')
ax.set_title('Orbit uniformity vs. empirical shift\nMar: high spread (orbit-dependent) | Apr: low spread (uniform)')
ax.grid(True, alpha=0.3)

# d_emp_std across dates
ax = axes[0, 1]
ax.bar(by_date.index, by_date['d_emp_std'], color=colors, edgecolor='black', lw=0.5)
ax.set_ylabel('d_empirical std')
ax.set_title('Scene variance — Mar & Apr have 2x the noise')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

# Fraction of scene < -1.0 vs > 1.0
ax = axes[1, 0]
x = np.arange(len(by_date))
w = 0.35
ax.bar(x - w/2, by_date['frac_gt_1'] * 100, w, label='> 1.0 (bright)', color='tab:orange', edgecolor='black', lw=0.5)
ax.bar(x + w/2, by_date['frac_lt_m1'] * 100, w, label='< -1.0 (drop)', color='tab:blue', edgecolor='black', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(by_date.index, rotation=45)
ax.set_ylabel('% of scene')
ax.set_title('Fraction of scene with extreme d_empirical\nApr: 47% drops | Feb 2024: 23% bright')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Skew
ax = axes[1, 1]
skew = df.groupby('date')['d_emp_skew'].mean()
skew_colors = ['tab:red' if s < -0.5 else 'tab:green' if s > 0.2 else 'tab:gray' for s in skew]
ax.bar(skew.index, skew, color=skew_colors, edgecolor='black', lw=0.5)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_ylabel('Skewness')
ax.set_title('d_empirical skewness\nNegative skew = tail of drops | Positive = tail of bright signals')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Two failure modes: wet snow (uniform) vs storm loading (orbit-dependent)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Per-Orbit Breakdown Across All Dates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, zone in zip(axes.flat, sorted(zone_short.keys())):
    zdf = df[df['zone'] == zone].set_index('date')
    orbit_data = pd.DataFrame({
        f'Orbit 71 VV': zdf.get('d_71_VV_emp_mean'),
        f'Orbit 93 VV': zdf.get('d_93_VV_emp_mean'),
        f'Orbit 173 VV': zdf.get('d_173_VV_emp_mean'),
    })
    orbit_data.plot(kind='bar', ax=ax, edgecolor='black', lw=0.5, width=0.75)
    ax.axhline(0, color='black', ls='-', lw=0.5)
    ax.set_title(zone_short[zone], fontsize=11)
    ax.set_ylabel('VV empirical mean')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.2, axis='y')

fig.suptitle('Per-orbit VV empirical distances by zone — Soldier amplifies orbit differences', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. The Soldier Zone Amplification Effect

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# d_emp by zone
ax = axes[0, 0]
pivot = df.pivot_table(values='d_emp_mean', index='date', columns='zone_short')
pivot.plot(ax=ax, marker='o', lw=1.5)
ax.axhline(0, color='black', ls='--', lw=0.5)
ax.set_ylabel('d_empirical mean')
ax.set_title('Empirical: Soldier most extreme\nin both directions')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# combined_distance by zone
ax = axes[0, 1]
pivot = df.pivot_table(values='d_comb_mean', index='date', columns='zone_short')
pivot.plot(ax=ax, marker='o', lw=1.5)
ax.axhline(0, color='black', ls='--', lw=0.5)
ax.set_ylabel('combined_distance mean')
ax.set_title('Combined: Soldier still most extreme\nbut gap narrows on good dates')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Orbit spread by zone
ax = axes[1, 0]
pivot = df.pivot_table(values='orbit_emp_VV_spread', index='date', columns='zone_short')
pivot.plot(ax=ax, marker='o', lw=1.5)
ax.set_ylabel('Orbit VV spread')
ax.set_title('Soldier has the largest orbit spreads\n(most orbit-dependent)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Mahalanobis by zone
ax = axes[1, 1]
pivot = df.pivot_table(values='d_mah_mean', index='date', columns='zone_short')
pivot.plot(ax=ax, marker='o', lw=1.5)
ax.set_ylabel('Mahalanobis mean')
ax.set_title('Mahalanobis also highest for Soldier on Feb 2024\n(real signal, not just bias)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Detection Counts and the 2024-02-05 Event

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Detection count by zone and date
ax = axes[0, 0]
pivot = df.pivot_table(values='det_count', index='date', columns='zone_short', aggfunc='sum')
pivot.plot(kind='bar', ax=ax, stacked=True, edgecolor='black', lw=0.3)
ax.set_ylabel('Detection pixel count')
ax.set_title('Detection counts — Feb 5, 2024 dominates')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.2, axis='y')

# Detection fraction vs d_empirical
ax = axes[0, 1]
for zone, grp in df.groupby('zone_short'):
    ax.scatter(grp['d_emp_mean'], grp['det_frac'] * 100, s=50, label=zone,
               alpha=0.8, edgecolors='black', lw=0.5)
ax.set_xlabel('d_empirical mean')
ax.set_ylabel('Detection fraction (%)')
ax.set_title('More positive empirical distance\n= more detections')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Detection fraction vs combined_distance
ax = axes[1, 0]
for zone, grp in df.groupby('zone_short'):
    ax.scatter(grp['d_comb_mean'], grp['det_frac'] * 100, s=50, label=zone,
               alpha=0.8, edgecolors='black', lw=0.5)
ax.set_xlabel('combined_distance mean')
ax.set_ylabel('Detection fraction (%)')
ax.set_title('combined_distance also correlates\nwith detection rate')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Detection fraction vs Mahalanobis
ax = axes[1, 1]
for zone, grp in df.groupby('zone_short'):
    ax.scatter(grp['d_mah_mean'], grp['det_frac'] * 100, s=50, label=zone,
               alpha=0.8, edgecolors='black', lw=0.5)
ax.set_xlabel('Mahalanobis mean')
ax.set_ylabel('Detection fraction (%)')
ax.set_title('Mahalanobis also correlates\nwith detection rate')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Detection Elevation Shifts with Season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

elev_data = df.dropna(subset=['det_elev_mean'])

ax = axes[0]
for zone, grp in elev_data.groupby('zone_short'):
    ax.scatter(grp['month'], grp['det_elev_mean'], s=60, label=zone,
               alpha=0.8, edgecolors='black', lw=0.5)
ax.set_xlabel('Month')
ax.set_ylabel('Mean detection elevation (m)')
ax.set_title('Detections migrate uphill through the season\nDec ~2100m | Feb-Apr ~2400-2560m')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xticks([1, 2, 3, 4, 12])
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'Dec'])

ax = axes[1]
for zone, grp in elev_data.groupby('zone_short'):
    ax.scatter(grp['swe_at_date'], grp['det_elev_mean'], s=60, label=zone,
               alpha=0.8, edgecolors='black', lw=0.5)
ax.set_xlabel('SWE at date (in)')
ax.set_ylabel('Mean detection elevation (m)')
ax.set_title('More SWE = higher detection elevation')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. VV vs VH: Polarization Differences

In [ ]:
# Compute VV-VH difference for each orbit across all zone-dates
vv_vh_data = []
for _, r in df.iterrows():
    for orbit in ['71', '93', '173']:
        vv = r.get(f'd_{orbit}_VV_emp_mean')
        vh = r.get(f'd_{orbit}_VH_emp_mean')
        if pd.notna(vv) and pd.notna(vh):
            vv_vh_data.append({
                'date': r['date'], 'zone': r['zone_short'], 'orbit': orbit,
                'VV': vv, 'VH': vh, 'VV_minus_VH': vv - vh,
            })
vvvh = pd.DataFrame(vv_vh_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VV-VH diff by date
ax = axes[0]
pivot = vvvh.pivot_table(values='VV_minus_VH', index='date', columns='orbit')
pivot.plot(kind='bar', ax=ax, edgecolor='black', lw=0.5)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_ylabel('VV - VH (empirical mean)')
ax.set_title('VV consistently drops more than VH\n(negative = VV more affected)')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Orbit')
ax.grid(True, alpha=0.2, axis='y')

# Apr 1 is the exception — smallest VV-VH gap
ax = axes[1]
by_date_vvvh = vvvh.groupby('date')['VV_minus_VH'].mean()
colors = ['tab:red' if d in ['2023-04-01'] else 'tab:blue' for d in by_date_vvvh.index]
ax.bar(by_date_vvvh.index, by_date_vvvh, color=colors, edgecolor='black', lw=0.5)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_ylabel('Mean VV - VH difference')
ax.set_title('Apr 1 (red): smallest VV-VH gap\nWet snow attenuates both polarizations equally')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

## 9. SNOTEL Context for All Dates

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Full SNOTEL record with all avalanche dates marked
avl_dates = [pd.Timestamp(d) for d in dates]
# Color by d_emp: red if negative, blue if positive
date_d_emp = df.groupby('date')['d_emp_mean'].mean()

# Temperature
ax = axes[0]
ax.fill_between(snotel.index, snotel['Tmin_C'], snotel['Tmax_C'],
                alpha=0.15, color='tab:red')
ax.plot(snotel.index, snotel['Tavg_C'], 'tab:red', lw=0.5, alpha=0.7)
ax.axhline(0, color='blue', ls='--', lw=0.5, alpha=0.4)
for d in avl_dates:
    d_str = d.strftime('%Y-%m-%d')
    color = 'red' if date_d_emp.get(d_str, 0) < -0.2 else 'green'
    ax.axvline(d, color=color, ls='--', lw=1.5, alpha=0.7)
ax.set_ylabel('Temperature (C)')
ax.set_title('Galena Summit SNOTEL — all high-danger dates\nGreen = positive d_emp (good detection) | Red = negative (biased)', fontsize=11)
ax.grid(True, alpha=0.2)

# SWE
ax = axes[1]
ax.plot(snotel.index, snotel['SWE_in'], 'tab:blue', lw=1.5)
for d in avl_dates:
    d_str = d.strftime('%Y-%m-%d')
    color = 'red' if date_d_emp.get(d_str, 0) < -0.2 else 'green'
    ax.axvline(d, color=color, ls='--', lw=1.5, alpha=0.7)
ax.set_ylabel('SWE (in)')
ax.set_title('SWE — avalanche dates at high SWE have negative empirical bias', fontsize=11)
ax.grid(True, alpha=0.2)

# Snow depth
ax = axes[2]
ax.plot(snotel.index, snotel['SnowDepth_in'], 'tab:purple', lw=1.5)
for d in avl_dates:
    d_str = d.strftime('%Y-%m-%d')
    color = 'red' if date_d_emp.get(d_str, 0) < -0.2 else 'green'
    ax.axvline(d, color=color, ls='--', lw=1.5, alpha=0.7)
ax.set_ylabel('Snow Depth (in)')
ax.set_title('Snow depth', fontsize=11)
ax.grid(True, alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 10. Cross-Date Stability: Empirical vs Combined vs Mahalanobis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

by_date_emp = df.groupby('date')['d_emp_mean'].mean()
by_date_comb = df.groupby('date')['d_comb_mean'].mean()
by_date_mah = df.groupby('date')['d_mah_mean'].mean()

# Side-by-side bars for all three
ax = axes[0]
x = np.arange(len(dates))
w = 0.25
ax.bar(x - w, by_date_emp, w, label='d_empirical', color='tab:blue', edgecolor='black', lw=0.5)
ax.bar(x, by_date_comb, w, label='combined_distance', color='tab:green', edgecolor='black', lw=0.5)
ax.bar(x + w, by_date_mah, w, label='Mahalanobis', color='tab:orange', edgecolor='black', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(dates, rotation=45)
ax.axhline(0, color='black', ls='-', lw=0.5)
ax.set_ylabel('Mean distance')
ax.set_title(f'All three metrics across dates\nEmpirical [{by_date_emp.min():.2f}, {by_date_emp.max():.2f}]  '
             f'Combined [{by_date_comb.min():.2f}, {by_date_comb.max():.2f}]  '
             f'Mahalanobis [{by_date_mah.min():.2f}, {by_date_mah.max():.2f}]')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

# Coefficient of variation
ax = axes[1]
emp_cv = by_date_emp.std() / abs(by_date_emp.mean()) * 100
comb_cv = by_date_comb.std() / by_date_comb.mean() * 100
mah_cv = by_date_mah.std() / by_date_mah.mean() * 100
bars = ax.bar(['Empirical', 'Combined', 'Mahalanobis'], [emp_cv, comb_cv, mah_cv],
       color=['tab:blue', 'tab:green', 'tab:orange'], edgecolor='black', lw=0.5)
ax.set_ylabel('Coefficient of Variation (%)')
ax.set_title('Cross-date variability\nCombined is between empirical and Mahalanobis')
ax.grid(True, alpha=0.2, axis='y')
for bar, v in zip(bars, [emp_cv, comb_cv, mah_cv]):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, f'{v:.0f}%', ha='center', fontsize=11)

# Correlation between SWE and each metric
ax = axes[2]
metrics = {
    'd_empirical': ('swe_at_date', 'd_emp_mean'),
    'combined_distance': ('swe_at_date', 'd_comb_mean'),
    'Mahalanobis': ('swe_at_date', 'd_mah_mean'),
}
r_vals = []
for name, (wx, sy) in metrics.items():
    valid = df[[wx, sy]].dropna()
    r_vals.append(valid[wx].corr(valid[sy]))
bars = ax.bar(list(metrics.keys()), [-r for r in r_vals],
       color=['tab:blue', 'tab:green', 'tab:orange'], edgecolor='black', lw=0.5)
ax.set_ylabel('|Correlation with SWE|')
ax.set_title('SWE bias strength\nML blending reduces but does not eliminate')
ax.grid(True, alpha=0.2, axis='y')
for bar, r in zip(bars, r_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, -r + 0.02, f'r={r:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 11. Sigma (ML Learned Uncertainty) Patterns

In [ ]:
# Do the ML sigma (learned noise) values correlate with weather?
sigma_data = []
for _, r in df.iterrows():
    for orbit in ['71', '93', '173']:
        for pol in ['VV', 'VH']:
            col = f'sigma_{orbit}_{pol}_ml_mean'
            if col in r and pd.notna(r[col]):
                sigma_data.append({
                    'date': r['date'], 'zone': r['zone_short'],
                    'orbit': orbit, 'pol': pol,
                    'sigma': r[col],
                    'd_emp_mean': r['d_emp_mean'],
                    'swe': r.get('swe_at_date', np.nan),
                })
sigma_df = pd.DataFrame(sigma_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigma by date
ax = axes[0]
for key, grp in sigma_df.groupby(['orbit', 'pol']):
    by_date_sigma = grp.groupby('date')['sigma'].mean()
    ls = '-' if key[1] == 'VV' else '--'
    ax.plot(by_date_sigma.index, by_date_sigma.values, marker='o', ls=ls,
            lw=1.2, markersize=4, label=f'Orbit {key[0]} {key[1]}')
ax.set_ylabel('Mean sigma (learned noise)')
ax.set_title('ML learned per-pixel noise across dates')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# Sigma vs SWE
ax = axes[1]
vv_sigma = sigma_df[sigma_df['pol'] == 'VV']
for orbit, grp in vv_sigma.groupby('orbit'):
    ax.scatter(grp['swe'], grp['sigma'], s=40, label=f'Orbit {orbit}', alpha=0.7)
ax.set_xlabel('SWE (in)')
ax.set_ylabel('Sigma VV (learned noise)')
ax.set_title('Sigma vs SWE — higher SWE = slightly lower sigma?')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Summary Table

In [ ]:
summary = df.groupby('date').agg({
    'd_emp_mean': 'mean',
    'd_comb_mean': 'mean',
    'd_mah_mean': 'mean',
    'd_emp_std': 'mean',
    'd_emp_skew': 'mean',
    'orbit_emp_VV_spread': 'mean',
    'det_count': 'sum',
    'det_elev_mean': 'mean',
    'tavg_window_mean': 'first',
    'tmax_window_max': 'first',
    'days_tmax_above_0': 'first',
    'swe_at_date': 'first',
    'depth_at_date': 'first',
    'swe_change_pre': 'first',
    'month': 'first',
}).round(2)

summary = summary.rename(columns={
    'd_emp_mean': 'd_emp',
    'd_comb_mean': 'd_comb',
    'd_mah_mean': 'd_mah',
    'd_emp_std': 'emp_std',
    'd_emp_skew': 'emp_skew',
    'orbit_emp_VV_spread': 'orb_spread',
    'det_count': 'det_total',
    'det_elev_mean': 'det_elev',
    'tavg_window_mean': 'Tavg',
    'tmax_window_max': 'Tmax',
    'days_tmax_above_0': 'warm_days',
    'swe_at_date': 'SWE',
    'depth_at_date': 'depth',
    'swe_change_pre': 'dSWE',
})

summary.style.background_gradient(subset=['d_emp'], cmap='RdBu', vmin=-1.5, vmax=1.5) \
    .background_gradient(subset=['d_comb'], cmap='RdYlGn', vmin=0, vmax=2) \
    .background_gradient(subset=['d_mah'], cmap='YlOrRd') \
    .background_gradient(subset=['det_total'], cmap='Greens') \
    .background_gradient(subset=['SWE'], cmap='Blues') \
    .format(precision=2)